In [1]:
import pytest
import ipytest
from pv import *

In [2]:
ipytest.autoconfig()

In [4]:
%%ipytest

@pytest.fixture
def base_pv_setup():
    """Initializes a PV object with consistent parameters."""
    Gsig = np.array([10.0])
    Gamb = np.array([500.0])
    # Area = 1e-4 m^2 (1 cm^2). Jsc = 35 mA/cm^2 -> Isc_ref = 35 mA
    params = {
        'A': 1e-4, 
        'Jsc': 35e-3, 
        'Rs': 0.5, 
        'Rsh': 500, 
        'Co': 1e-6, # AC coupling cap
        'Lo': 1e-3, # Energy harvesting inductor
        'Rc': 10
    }
    return PV(Gsig, Gamb, **params)

def test_pv_dc_scaling(base_pv_setup):
    """Verify Iph scales correctly with irradiance."""
    pv = base_pv_setup
    # Total G = 510. Reference G = 1000.
    # Ref Isc = Jsc * (A * 1e4) = 0.035 * 1 = 0.035A
    # Expected Iph = 0.035 * (510/1000) = 0.01785A
    expected_iph = 0.035 * (510 / 1000.0)
    assert pv.Iph[0] == pytest.approx(expected_iph, rel=1e-3)

def test_transfer_function_bandpass_behavior(base_pv_setup):
    """
    Verify the transfer function behavior. 
    Because of Co (AC coupling), the gain is LOW at DC and peaks at mid-freq.
    """
    pv = base_pv_setup
    # hpv is (N_pv, 1, Nfreq)
    # Check gain at 100Hz vs gain at a higher frequency (e.g., mid-sweep)
    gain_start = pv.hpv[0, 0, 0]
    gain_max = np.max(pv.hpv[0, 0, :])
    
    # The AC coupling capacitor should block low frequencies
    assert gain_max > gain_start
    # Ensure the gain is physically significant somewhere in the sweep
    assert gain_max > 0.01 

def test_lambert_w_at_boundaries(base_pv_setup):
    """Verify IV curve at SC and OC boundaries."""
    pv = base_pv_setup
    # Current at 0V should be roughly Iph
    assert pv.pv_current(0)[0] == pytest.approx(pv.Iph[0], rel=1e-2)
    # Current at Voc should be 0
    assert pv.pv_current(pv.Voc)[0] == pytest.approx(0, abs=1e-4)

def test_shot_noise_positivity(base_pv_setup):
    """Verify shot noise exists and is positive."""
    pv = base_pv_setup
    assert pv.sh_noise[0, 0] > 0

....                                                                                         [100%]
4 passed in 0.19s
